In [1]:
# (Optional) Install extra libraries if needed
# !pip install --quiet seaborn scikit-learn matplotlib pandas numpy

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error

from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.style.use('seaborn-v0_8')
sns.set_context('talk')
pd.set_option('display.max_columns',200)

In [35]:
DATA_PATH = '.'  # e.g., './data'
FILE_NAME = 'E0.csv'  # <-- update to the actual file, e.g., 'Metro_Interstate_Traffic_Volume.csv'

# Try to read the CSV (adjust encoding or separator if needed)
file_path = os.path.join(DATA_PATH, FILE_NAME)
assert os.path.exists(file_path), f"File not found: {file_path}. Please place the dataset and update FILE_NAME."

df = pd.read_csv(file_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.tail()

Shape: (240, 132)
Columns: ['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BFDH', 'BFDD', 'BFDA', 'BMGMH', 'BMGMD', 'BMGMA', 'BVH', 'BVD', 'BVA', 'BWH', 'BWD', 'BWA', 'CLH', 'CLD', 'CLA', 'LBH', 'LBD', 'LBA', 'PSH', 'PSD', 'PSA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'BFEH', 'BFED', 'BFEA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'BFE>2.5', 'BFE<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'BFEAHH', 'BFEAHA', 'B365CH', 'B365CD', 'B365CA', 'BFDCH', 'BFDCD', 'BFDCA', 'BMGMCH', 'BMGMCD', 'BMGMCA', 'BVCH', 'BVCD', 'BVCA', 'BWCH', 'BWCD', 'BWCA', 'CLCH', 'CLCD', 'CLCA', 'LBCH', 'LBCD', 'LBCA', 'PSCH', 'PSCD', 'PSCA', 'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA', 'BFECH', 'BFECD', 'BFECA', 'B365C>2.5', 'B365C<2.5', 'PC>2.5', 'P

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A,BFDH,BFDD,BFDA,BMGMH,BMGMD,BMGMA,BVH,BVD,BVA,BWH,BWD,BWA,CLH,CLD,CLA,LBH,LBD,LBA,PSH,PSD,PSA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,BFEH,BFED,BFEA,B365>2.5,B365<2.5,P>2.5,P<2.5,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,BFE>2.5,BFE<2.5,AHh,B365AHH,B365AHA,PAHH,PAHA,MaxAHH,MaxAHA,AvgAHH,AvgAHA,BFEAHH,BFEAHA,B365CH,B365CD,B365CA,BFDCH,BFDCD,BFDCA,BMGMCH,BMGMCD,BMGMCA,BVCH,BVCD,BVCA,BWCH,BWCD,BWCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA,PSCH,PSCD,PSCA,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,AvgCA,BFECH,BFECD,BFECA,B365C>2.5,B365C<2.5,PC>2.5,PC<2.5,MaxC>2.5,MaxC<2.5,AvgC>2.5,AvgC<2.5,BFEC>2.5,BFEC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
235,E0,01/02/2026,14:00,Aston Villa,Brentford,0,1,A,0,1,A,T Robinson,27,6,5,2,8,8,12,1,0,3,0,0,2.10,3.5,3.40,2.10,3.50,3.50,2.08,3.55,3.55,2.05,3.6,3.40,2.05,3.5,3.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.12,3.6,3.60,2.08,3.47,3.46,2.16,3.65,3.65,1.80,2.00,NaN,NaN,1.80,2.06,1.78,1.98,1.88,2.10,-0.25,1.83,2.03,NaN,NaN,1.83,2.03,1.79,1.99,1.86,2.12,2.38,3.40,3.00,2.30,3.40,3.20,2.28,3.35,3.30,2.30,3.30,3.13,2.25,3.40,3.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.38,3.4,3.30,2.30,3.32,3.13,2.38,3.45,3.40,1.88,1.98,NaN,NaN,1.91,2.00,1.85,1.92,1.94,2.04,-0.25,2.05,1.80,NaN,NaN,2.05,1.88,1.95,1.83,2.03,1.95
236,E0,01/02/2026,14:00,Man United,Fulham,3,2,H,1,0,H,J Brooks,13,14,6,6,6,9,3,7,2,1,0,0,1.57,4.5,5.00,1.60,4.33,5.50,1.57,4.50,5.50,1.57,4.2,5.50,1.61,4.2,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.61,4.6,5.50,1.58,4.32,5.20,1.62,4.70,5.80,1.67,2.20,NaN,NaN,1.67,2.25,1.65,2.17,1.72,2.36,-1.00,1.98,1.88,NaN,NaN,2.00,1.88,1.95,1.82,2.04,1.92,1.53,4.75,5.25,1.53,4.60,5.50,1.52,4.80,5.60,1.50,4.60,5.75,1.53,4.75,5.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.55,4.8,5.75,1.52,4.63,5.47,1.57,4.90,6.20,1.50,2.63,NaN,NaN,1.50,2.75,1.47,2.62,1.52,2.88,-1.00,1.85,2.00,NaN,NaN,1.85,2.00,1.81,1.96,1.92,2.07
237,E0,01/02/2026,14:00,Nott'm Forest,Crystal Palace,1,1,D,1,1,D,M Salisbury,9,14,1,3,4,12,5,7,1,3,0,0,1.95,3.5,3.90,1.95,3.50,4.00,1.97,3.55,4.00,1.95,3.5,3.90,1.98,3.5,3.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.98,3.6,4.00,1.95,3.48,3.88,2.02,3.65,4.10,2.00,1.80,NaN,NaN,2.04,1.80,1.97,1.78,2.14,1.86,-0.50,1.98,1.88,NaN,NaN,1.98,1.90,1.90,1.83,2.03,1.96,2.10,3.30,3.60,2.15,3.25,3.75,2.12,3.30,3.80,2.10,3.13,3.80,2.05,3.30,3.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.15,3.3,4.00,2.09,3.22,3.73,2.16,3.40,4.00,2.20,1.67,NaN,NaN,2.22,1.76,2.16,1.68,2.30,1.76,-0.25,1.83,2.03,NaN,NaN,1.83,2.05,1.78,2.02,1.84,2.16
238,E0,01/02/2026,16:30,Tottenham,Man City,2,2,D,0,2,A,R Jones,12,15,6,3,15,12,3,4,3,3,0,0,4.50,4.1,1.73,4.60,4.00,1.73,4.60,4.35,1.68,4.40,4.0,1.67,4.20,4.1,1.73,4.2,4.0,1.75,4.2,4.0,1.73,NaN,NaN,NaN,4.60,4.4,1.75,4.41,4.06,1.71,4.80,4.30,1.77,1.62,2.30,NaN,NaN,1.62,2.32,1.61,2.26,1.68,2.42,0.75,1.93,1.93,NaN,NaN,1.95,1.93,1.91,1.87,2.01,1.97,4.75,4.10,1.67,5.00,4.20,1.70,5.10,4.10,1.66,5.50,4.00,1.62,4.60,4.00,1.68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.50,4.2,1.70,4.98,4.01,1.65,5.40,4.20,1.72,1.62,2.30,NaN,NaN,1.65,2.43,1.61,2.27,1.68,2.44,0.75,2.03,1.83,NaN,NaN,2.03,1.83,1.99,1.79,2.08,1.90
239,E0,02/02/2026,20:00,Sunderland,Burnley,3,0,H,2,0,H,P Tierney,14,5,5,0,12,10,1,2,1,4,0,0,1.80,3.5,4.75,1.80,3.50,5.00,1.81,3.50,4.80,1.80,3.4,4.60,1.78,3.6,4.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.83,3.6,5.00,1.80,3.42,4.73,1.86,3.65,5.00,2.20,1.67,NaN,NaN,2.28,1.67,2.21,1.63,2.34,1.70,-0.75,2.03,1.78,NaN,NaN,2.05,1.83,2.02,1.75,2.13,1.83,1.91,3.30,4.33,1.91,3.30,4.50,1.94,3.20,4.70,1.85,3.30,4.75,1.93,3.25,4.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.95,3.3,4.75,1.91,3.23,4.46,2.00,3.45,4.60,2.50,1.53,NaN,NaN,2.50,1.57,2.42,1.54,2.62,1.61,-0.50,1.95,1.90,NaN,NaN,1.95,1.90,1.86,1.87,1.98,2.01


In [36]:
df["Yellowcards"]= df["HY"] + df["AY"]

In [37]:
# Group by referee and sum yellows
ref_yellows = df.groupby("Referee")["Yellowcards"].sum()

# Sort descending and show top 10
top_refs = ref_yellows.sort_values(ascending=False).head(10)
print(top_refs)

Referee
A Taylor      75
C Kavanagh    71
S Attwell     67
P Bankes      65
S Hooper      58
R Jones       55
D England     53
S Barrott     53
M Oliver      51
T Bramall     48
Name: Yellowcards, dtype: int64


In [38]:
# 2️⃣ Find the index of the match with the most yellows
max_yellow_idx = df["Yellowcards"].idxmax()

# 3️⃣ Get the match info
most_yellow_match = df.loc[max_yellow_idx]

print(most_yellow_match[["Date", "HomeTeam", "AwayTeam", "HY", "AY", "Yellowcards"]])

Date            29/11/2025
HomeTeam        Sunderland
AwayTeam       Bournemouth
HY                       5
AY                       5
Yellowcards             10
Name: 122, dtype: object


In [39]:
df["TotalReds"] = df["HR"] + df["AR"]

In [40]:
max_red_idc = df["TotalReds"].idxmax()

most_red = df.loc[max_red_idc]

print(most_red[["Date", "HomeTeam", "AwayTeam", "HR", "AR"]])

Date        20/09/2025
HomeTeam    Man United
AwayTeam       Chelsea
HR                   1
AR                   1
Name: 45, dtype: object


In [41]:
ref_yellows = df.groupby("Referee")["TotalReds"].sum()

# Sort descending and show top 10
top_refs = ref_yellows.sort_values(ascending=False).head(10)
print(top_refs)

Referee
J Brooks      3
P Bankes      3
A Taylor      2
D England     2
T Kirk        2
S Hooper      2
T Robinson    1
S Attwell     1
C Kavanagh    1
C Pawson      1
Name: TotalReds, dtype: int64


In [53]:
team_name = "Arsenal"  # replace with your team
home_matches = df[df["HomeTeam"] == team_name]
total_home_goals = home_matches["FTHG"].sum()
total_home_conceded = home_matches["FTAG"].sum()
away_matches= df[df["AwayTeam"] ==team_name]
total_away_goals = away_matches["FTAG"].sum()
total_away_conceded = away_matches["FTHG"].sum()

print(f"{team_name} scored {total_home_goals} at home.")
print(f"{team_name} conceded {total_home_conceded} at home.")
print(f"{team_name} scored {total_away_goals} away.")
print(f"{team_name} conceded {total_away_conceded} away.")

Arsenal scored 28 at home.
Arsenal conceded 8 at home.
Arsenal scored 18 away.
Arsenal conceded 9 away.


In [54]:
total_goals = total_home_goals + total_away_goals
total_against_goals = total_home_conceded + total_away_conceded

In [55]:
print(f"{team_name} scored {total_goals} goals in total.")
print(f"{team_name} conceded {total_against_goals} goals in total.")

Arsenal scored 46 goals in total.
Arsenal conceded 17 goals in total.


Arsenal conceded 17 goals in total.
